In this example we create an Embedding layer for a small vocabulary of four words ("king", "queen", "man", "woman"). We set the embeddings manually so that the vector arithmetic king - man + woman equals the embedding of queen. Running the code shows that the computed vector matches the predefined queen embedding.

Reference:
1. [Embedding Layer](https://www.youtube.com/watch?v=aWFllV6WsAs)

In [26]:
import torch
import torch.nn as nn

# Create a small vocabulary: {0: padding, 1: "king", 2: "queen", 3: "man", 4: "woman"}
vocab_size = 5
embedding_dim = 3
embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

# Manually set the embeddings so that:
# embedding("king") - embedding("man") + embedding("woman") = embedding("queen")
# Let's define:
# king  = [1.0, 0.0, 1.0]
# man   = [1.0, 0.0, 0.0]
# woman = [0.0, 1.0, 0.0]
# Then, king - man + woman = [0.0, 1.0, 1.0], so we set queen = [0.0, 1.0, 1.0]

# Manually assign weights
with torch.no_grad():
  embedding.weight[0] # padding, it's already zero
  embedding.weight[1] = torch.tensor([1.0, 0.0, 1.0])  # king
  embedding.weight[2] = torch.tensor([0.0, 1.0, 1.0])  # queen
  embedding.weight[3] = torch.tensor([1.0, 0.0, 0.0])  # man
  embedding.weight[4] = torch.tensor([0.0, 1.0, 0.0])  # woman

# Get embeddings for the words (using the assigned indices)
king   = embedding(torch.tensor(1))
queen  = embedding(torch.tensor(2))
man    = embedding(torch.tensor(3))
woman  = embedding(torch.tensor(4))

# Compute the arithmetic: king - man + woman
result = king - man + woman

print("Embedding for king:   ", king)
print("Embedding for man:    ", man)
print("Embedding for woman:  ", woman)
print("Computed queen:       ", result)
print("Actual embedding queen:", queen)

x = embedding(torch.tensor([[1, 2, 0], # "king queen <pad>"
                            [1, 0, 0], # "king <pad> <pad>"
                           ]))
print(f"x.shape: {x.shape}")
print(f"Embedding for x:\n{x}")

Embedding for king:    tensor([1., 0., 1.], grad_fn=<EmbeddingBackward0>)
Embedding for man:     tensor([1., 0., 0.], grad_fn=<EmbeddingBackward0>)
Embedding for woman:   tensor([0., 1., 0.], grad_fn=<EmbeddingBackward0>)
Computed queen:        tensor([0., 1., 1.], grad_fn=<AddBackward0>)
Actual embedding queen: tensor([0., 1., 1.], grad_fn=<EmbeddingBackward0>)
x.shape: torch.Size([2, 3, 3])
Embedding for x:
tensor([[[1., 0., 1.],
         [0., 1., 1.],
         [0., 0., 0.]],

        [[1., 0., 1.],
         [0., 0., 0.],
         [0., 0., 0.]]], grad_fn=<EmbeddingBackward0>)


# Embedding layer training demo

In [28]:
import torch

import torch.nn as nn
import torch.optim as optim

# Create a small vocabulary: {0: padding, 1: "king", 2: "queen", 3: "man", 4: "woman"}
vocab_size = 5
embedding_dim = 3
embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)

# Optimizer and loss function
optimizer = optim.SGD(embedding.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

# Train to enforce: embedding("king") - embedding("man") + embedding("woman") ~ embedding("queen")
num_epochs = 100
for epoch in range(num_epochs):
  optimizer.zero_grad()
  
  king = embedding(torch.tensor(1))
  queen = embedding(torch.tensor(2))
  man = embedding(torch.tensor(3))
  woman = embedding(torch.tensor(4))
  
  predicted = king - man + woman
  loss = loss_fn(predicted, queen)
  loss.backward()
  optimizer.step()

  if epoch % 10 == 0:
    print(f"Epoch {epoch}, Loss: {loss.item()}")

# Get embeddings for the words (using the assigned indices)
king   = embedding(torch.tensor(1))
queen  = embedding(torch.tensor(2))
man    = embedding(torch.tensor(3))
woman  = embedding(torch.tensor(4))

# Compute the arithmetic: king - man + woman
result = king - man + woman

print("Embedding for king:   ", king)
print("Embedding for man:    ", man)
print("Embedding for woman:  ", woman)
print("Computed queen:       ", result)
print("Actual embedding queen:", queen)

Epoch 0, Loss: 3.73799204826355
Epoch 10, Loss: 0.0075625148601830006
Epoch 20, Loss: 1.5299714505090378e-05
Epoch 30, Loss: 3.095631129212961e-08
Epoch 40, Loss: 6.1628945002834e-11
Epoch 50, Loss: 1.6862437824099324e-13
Epoch 60, Loss: 1.7541523789077473e-14
Epoch 70, Loss: 1.541359576052396e-14
Epoch 80, Loss: 1.541359576052396e-14
Epoch 90, Loss: 1.541359576052396e-14
Embedding for king:    tensor([ 0.1729, -1.0396,  0.0965], grad_fn=<EmbeddingBackward0>)
Embedding for man:     tensor([-0.3856, -1.5702,  0.2996], grad_fn=<EmbeddingBackward0>)
Embedding for woman:   tensor([-0.6621, -0.8117, -0.5575], grad_fn=<EmbeddingBackward0>)
Computed queen:        tensor([-0.1037, -0.2812, -0.7606], grad_fn=<AddBackward0>)
Actual embedding queen: tensor([-0.1037, -0.2812, -0.7606], grad_fn=<EmbeddingBackward0>)
